
%md
# Simple chain Evaluation

After creating the LLM endpoint called "claude_45". It s time to manipulate.

In this notebook, we will manipulate our first LLM and exploit MLflow tracing.


In [0]:
%pip install -U --quiet mlflow[databricks]==3.4.0 
dbutils.library.restartPython()


### Load the mlflow config 

In [0]:
%run ../_config/config_chain

## Call the model endpoint

We are using the model endpoint of the monolithic LLM we've registered, simple_chain



In [0]:
import mlflow
import mlflow.deployments
from typing import List, Dict

# Launch  a deploy client 
mlflow_client = mlflow.deployments.get_deploy_client("databricks")
endpoint_name = "claude_45"
# A few questions about the databricks mlflow system
DOCS_DATA = [
    {"question": "what are mlflow versions supported on databricks ?"},
    {"question": "what is a langchain mlflow flavor?"},
    {"question": "what is a scorer mlflow ?"},
    {"question": "how do I register a model ?"}
]


@mlflow.trace
def generate_docs(question: str) -> dict:
    """ this a wrapper on the predict function of the model to format the response."""
    system_instructions = "You are a helpful Databricks ML engineer. Answer the following question in a clear and concise manner. If you don't know the answer, just say that you don't know it, don't try to make up an answer."
    response = mlflow_client.predict(
        endpoint=endpoint_name,
        inputs={"messages": [
            {"role": "system", "content": system_instructions},
            {"role": "user", "content": question}
            ]}
    )

    return {"response": response["choices"][0]["message"]["content"]}

# Test the application
for q in DOCS_DATA : 
    print("response : ", generate_docs(q["question"]))

### search traces

In [0]:
import time

# Get traces from last five minutes
timestamp = int(time.time() * 1000)
chain_traces_last_5_min = mlflow.search_traces(filter_string=f"trace.status = 'OK' AND trace.timestamp_ms > {timestamp - 300000}")

# eval_traces is a Pandas DataFrame that has the evaluated traces.  The column `assessments` includes each scorer's feedback.
display(chain_traces_last_5_min)